# KrishiBazar AI - Odisha Mandi Data Exploration

This notebook explores the cleaned Odisha mandi-price data used for crop-price forecasting and mandi recommendations.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_orissa_mandi.csv'
pd.set_option('display.max_columns', 30)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Expected cleaned data at {DATA_PATH}. Run src/data/preprocessing.py first.')

In [ ]:
df = pd.read_csv(DATA_PATH)
df['price date'] = pd.to_datetime(df['price date'], errors='coerce')
df = df.sort_values(['commodity', 'market name', 'price date']).reset_index(drop=True)

print(f'Rows: {len(df):,}')
print(f'Date range: {df["price date"].min().date()} to {df["price date"].max().date()}')
df.head()

## Data quality

The forecast target is `modal_price`; usable records require a date, commodity, market, and positive target price.

In [ ]:
required = ['commodity', 'market name', 'district name', 'price date', 'modal_price']
quality = pd.DataFrame({
    'missing_values': df[required].isna().sum(),
    'missing_percent': (df[required].isna().mean() * 100).round(2),
})
print(f'Non-positive modal prices: {(df["modal_price"] <= 0).sum():,}')
print(f'Duplicate market/commodity/date rows: {df.duplicated(["market name", "commodity", "price date"]).sum():,}')
quality

In [ ]:
coverage = (
    df.groupby('commodity', observed=True)
      .agg(observations=('modal_price', 'size'), markets=('market name', 'nunique'),
           districts=('district name', 'nunique'), first_date=('price date', 'min'),
           last_date=('price date', 'max'), median_price=('modal_price', 'median'))
      .sort_values('observations', ascending=False)
)
coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.boxplot(column='modal_price', by='commodity', ax=axes[0], grid=False)
axes[0].set_title('Modal-price distribution by crop')
axes[0].set_xlabel('Commodity')
axes[0].set_ylabel('Price (INR/quintal)')
plt.suptitle('')

monthly = (df.assign(month=df['price date'].dt.to_period('M').dt.to_timestamp())
             .groupby(['month', 'commodity'], observed=True)['modal_price'].median().unstack())
monthly.plot(ax=axes[1], linewidth=2)
axes[1].set_title('Monthly median modal price')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Price (INR/quintal)')
axes[1].legend(title='Commodity')
plt.tight_layout()

In [ ]:
latest_date = df['price date'].max()
latest_snapshot = (df.loc[df['price date'].eq(latest_date)]
                   .groupby(['commodity', 'district name'], observed=True)
                   .agg(markets=('market name', 'nunique'), median_price=('modal_price', 'median'))
                   .sort_values(['commodity', 'median_price'], ascending=[True, False]))
print(f'Latest available date: {latest_date.date()}')
latest_snapshot.head(20)

## Next step

Run `02_feature_engineering.ipynb` to create calendar, lag, and rolling-window features without leaking a current price into its own prediction.